In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE WAREHOUSE CUSTOMER_SCD_WH
WITH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE DATABASE CUSTOMER_SCD_DB;

In [ ]:
%%sql -r dataframe_1
USE DATABASE CUSTOMER_SCD_DB;

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE SCHEMA SCD_SCHEMA;

In [ ]:
%%sql -r dataframe_5
USE SCHEMA SCD_SCHEMA;

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE FILE FORMAT CUSTOMER_CSV_FORMAT
TYPE = CSV
FIELD_DELIMITER = ','
SKIP_HEADER = 1
FIELD_OPTIONALLY_ENCLOSED_BY = '"'
NULL_IF = ('NULL', 'null', '');

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE STAGE CUSTOMER_STAGE
FILE_FORMAT = CUSTOMER_CSV_FORMAT;

In [ ]:
%%sql -r dataframe_8
LIST @CUSTOMER_STAGE;

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE TABLE DIM_CUSTOMER
(
    CUSTOMER_KEY NUMBER AUTOINCREMENT,
    CUSTOMER_ID NUMBER,
    CUSTOMER_NAME VARCHAR,
    CITY VARCHAR,
    STATE VARCHAR,
    MEMBERSHIP VARCHAR,
    SEGMENT VARCHAR
);

In [ ]:
%%sql -r dataframe_10
COPY INTO DIM_CUSTOMER
(
    CUSTOMER_ID,
    CUSTOMER_NAME,
    CITY,
    STATE,
    MEMBERSHIP,
    SEGMENT
)
FROM @CUSTOMER_STAGE/customers_initial.csv
FILE_FORMAT = (
    FORMAT_NAME = CUSTOMER_CSV_FORMAT
);

In [ ]:
%%sql -r dataframe_11
SELECT COUNT(*) AS TOTAL_CUSTOMERS
FROM DIM_CUSTOMER;

In [ ]:
%%sql -r dataframe_12
SELECT *
FROM DIM_CUSTOMER
ORDER BY CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_13
CREATE OR REPLACE TABLE CUSTOMER_UPDATES
(
    CUSTOMER_ID NUMBER,
    CUSTOMER_NAME VARCHAR,
    CITY VARCHAR,
    STATE VARCHAR,
    MEMBERSHIP VARCHAR,
    SEGMENT VARCHAR
);

In [ ]:
%%sql -r dataframe_14
COPY INTO CUSTOMER_UPDATES
FROM @CUSTOMER_STAGE/customer_updates.csv
FILE_FORMAT = (
    FORMAT_NAME = CUSTOMER_CSV_FORMAT
);

In [ ]:
%%sql -r dataframe_15
SELECT COUNT(*) AS RECORDS_RECEIVED
FROM CUSTOMER_UPDATES;

In [ ]:
%%sql -r dataframe_16
SELECT *
FROM CUSTOMER_UPDATES
ORDER BY CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_17
SELECT
    d.CUSTOMER_ID,
    d.CITY AS OLD_CITY,
    u.CITY AS NEW_CITY,
    d.MEMBERSHIP AS OLD_MEMBERSHIP,
    u.MEMBERSHIP AS NEW_MEMBERSHIP
FROM DIM_CUSTOMER d
JOIN CUSTOMER_UPDATES u
    ON d.CUSTOMER_ID = u.CUSTOMER_ID
WHERE
       NVL(d.CITY, '') <> NVL(u.CITY, '')
    OR NVL(d.STATE, '') <> NVL(u.STATE, '')
    OR NVL(d.MEMBERSHIP, '') <> NVL(u.MEMBERSHIP, '')
    OR NVL(d.SEGMENT, '') <> NVL(u.SEGMENT, '')
ORDER BY d.CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_18
SELECT
    d.CUSTOMER_ID,
    'CITY' AS ATTRIBUTE,
    d.CITY AS OLD_VALUE,
    u.CITY AS NEW_VALUE
FROM DIM_CUSTOMER d
JOIN CUSTOMER_UPDATES u
    ON d.CUSTOMER_ID = u.CUSTOMER_ID
WHERE NVL(d.CITY, '') <> NVL(u.CITY, '')

UNION ALL

SELECT
    d.CUSTOMER_ID,
    'STATE' AS ATTRIBUTE,
    d.STATE AS OLD_VALUE,
    u.STATE AS NEW_VALUE
FROM DIM_CUSTOMER d
JOIN CUSTOMER_UPDATES u
    ON d.CUSTOMER_ID = u.CUSTOMER_ID
WHERE NVL(d.STATE, '') <> NVL(u.STATE, '')

UNION ALL

SELECT
    d.CUSTOMER_ID,
    'MEMBERSHIP' AS ATTRIBUTE,
    d.MEMBERSHIP AS OLD_VALUE,
    u.MEMBERSHIP AS NEW_VALUE
FROM DIM_CUSTOMER d
JOIN CUSTOMER_UPDATES u
    ON d.CUSTOMER_ID = u.CUSTOMER_ID
WHERE NVL(d.MEMBERSHIP, '') <> NVL(u.MEMBERSHIP, '')

UNION ALL

SELECT
    d.CUSTOMER_ID,
    'SEGMENT' AS ATTRIBUTE,
    d.SEGMENT AS OLD_VALUE,
    u.SEGMENT AS NEW_VALUE
FROM DIM_CUSTOMER d
JOIN CUSTOMER_UPDATES u
    ON d.CUSTOMER_ID = u.CUSTOMER_ID
WHERE NVL(d.SEGMENT, '') <> NVL(u.SEGMENT, '')

ORDER BY CUSTOMER_ID, ATTRIBUTE;

In [ ]:
%%sql -r dataframe_19
UPDATE DIM_CUSTOMER d
SET
    CUSTOMER_NAME = u.CUSTOMER_NAME,
    CITY = u.CITY,
    STATE = u.STATE,
    MEMBERSHIP = u.MEMBERSHIP,
    SEGMENT = u.SEGMENT
FROM CUSTOMER_UPDATES u
WHERE d.CUSTOMER_ID = u.CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_20
SELECT
    CUSTOMER_ID,
    CUSTOMER_NAME,
    CITY,
    STATE,
    MEMBERSHIP,
    SEGMENT
FROM DIM_CUSTOMER
ORDER BY CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_21
SELECT
    CUSTOMER_ID,
    CUSTOMER_NAME,
    CITY,
    STATE,
    MEMBERSHIP
FROM DIM_CUSTOMER
WHERE CUSTOMER_ID = 101;

In [ ]:
%%sql -r dataframe_22
SELECT
    CUSTOMER_ID,
    CUSTOMER_NAME,
    CITY AS CURRENT_CITY,
    STATE AS CURRENT_STATE,
    MEMBERSHIP AS CURRENT_MEMBERSHIP
FROM DIM_CUSTOMER
WHERE CUSTOMER_ID = 101;